In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]


tau = length(Istar_obs)

model_tag_sym = :reciprocal

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 1

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [reciprocal_model] Fitting chain 1 (tau=34)
[ Info: [reciprocal] iter 1000/1000000 elapsed=3.9s, rate=0.168, mean=[1.815, 0.00116, 1.842, 0.203], std=[0.3596, 0.000269, 0.2033, 0.0457] [ADAPT]
[ Info: [reciprocal] iter 2000/1000000 elapsed=6.9s, rate=0.138, mean=[2.020, 0.00103, 1.923, 0.192], std=[0.3165, 0.000234, 0.1625, 0.0341] [ADAPT]
[ Info: [reciprocal] iter 3000/1000000 elapsed=9.1s, rate=0.127, mean=[2.084, 0.00098, 1.925, 0.191], std=[0.2775, 0.000210, 0.1410, 0.0300] [ADAPT]
[ Info: [reciprocal] iter 4000/1000000 elapsed=11.2s, rate=0.116, mean=[2.127, 0.00093, 1.895, 0.199], std=[0.2533, 0.000202, 0.1483, 0.0301] [ADAPT]
[ Info: [reciprocal] iter 5000/1000000 elapsed=13.4s, rate=0.111, mean=[2.159, 0.00091, 1.872, 0.200], std=[0.2356, 0.000191, 0.1403, 0.0278] [ADAPT]
[ Info: [reciprocal] iter 6000/1000000 elapsed=15.6s, rate=0.110, mean=[2.170, 0.00090, 1.872, 0.199], std=[0.2214, 0.000180, 0.1333, 0.0265] [ADAPT]
[ Info: [reciprocal] iter 7000/1000000 elapsed=17.7